# Day 7 — LightGBM：GBDTの実戦版

Day5までは sklearn の `GradientBoostingClassifier`。今日はKaggle/実務の本命 **LightGBM** へ。中身は同じGBDTだが「実戦仕様」。違いは3つの武器に集約される：

| 武器 | 何の話か | ひとこと |
|---|---|---|
| ① histogram | どのしきい値を候補にするか | 255箱に丸めて高速化（速さ） |
| ② leaf-wise | 木をどう育てるか | 一番得する葉だけ深掘り（best-first） |
| ③ early stopping | 木は何本積むか | validationを見張って自動で止める |

横串の核心：**LightGBMの強さは“生で賢い”じゃなく“速いから賢くできる”。速さが試行回数に化ける。**

In [ ]:
# 足場：import とデータ読み込み
import pandas as pd
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier

df = pd.read_csv('../../data/chat_sessions_complex.csv')
X = df.drop(columns=['deepened'])
y = df['deepened']
X.shape, y.value_counts().to_dict()

## 本筋①：初めてのLightGBM（生で回す）

お題：複雑データで cv=10、Day5 と公平に比較（同条件 n=300, lr=0.1）。

**学習者の予想：「LightGBM最強、めっちゃ上がる」**

### 結果
| モデル | cv精度 |
|---|---|
| sklearn GBDT（Day5, n=300, lr=0.1） | **0.806** |
| RF（Day4/5） | 0.795 |
| **LightGBM（今日, 同条件）** | **0.775** |

→ **予想は外れ。** 生のLightGBMは「精度の魔法」じゃない。同条件なら同じ土俵（小データではむしろやや下）。**精度のタダ飯は無い。**

In [ ]:
# 本筋①（学習者が記述）：Day5と同条件で cv=10
model = LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
scores = cross_val_score(model, X, y, cv=10)
print(f'LightGBM : {scores.mean():.3f} +/- {scores.std():.3f}')
# => LightGBM : 0.775 +/- 0.018

## 武器①：histogram ＝ なぜLightGBMを使うのか（速さ）

- **問題**：決定木は最良の分割を探すとき、列のしきい値を片っ端から試す（Day2）。候補数は最大で件数ぶん → 50万・100万行で激遅。
- **解**：木を建てる *前* に、各連続列を **255個の箱(bin)** に丸める。試すしきい値は常に **255通りに固定**。Nが増えても候補は増えない。
- **なぜ精度が落ちないか**：木はYES/NOの粗いカットしかしない。23.41歳と23.43歳で切っても同じ人が左右に分かれるだけ＝捨てた解像度に予測情報はほぼ無い。

下のセルで sklearn GBDT と速度＆精度を直接対決（10万行・足場）。

In [ ]:
# 足場：速度＆精度の直接対決（※ sklearn 側は重い：約95秒）
import time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification

Xb, yb = make_classification(n_samples=100_000, n_features=20, n_informative=10,
                             n_redundant=5, random_state=0)

g = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=0)
t = time.perf_counter(); g.fit(Xb, yb); t_gbdt = time.perf_counter() - t

l = LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1)
t = time.perf_counter(); l.fit(Xb, yb); t_lgbm = time.perf_counter() - t

print(f'sklearn GBDT : {t_gbdt:6.2f} 秒')
print(f'LightGBM     : {t_lgbm:6.2f} 秒')
print(f'LightGBM は約 {t_gbdt / t_lgbm:.1f} 倍速い')
# => sklearn 94.96秒 / LightGBM 1.11秒 / 約85.5倍。精度(cv=10)はGBDT0.786 vs LightGBM0.776 でほぼ互角。

## 武器②：leaf-wise ＝ 木の育て方

- **level-wise（sklearn GBDT）= 幅優先探索(BFS)**：木を「階」ごとに揃えて広げる。左右対称な木。
- **leaf-wise（LightGBM）= 最良優先(best-first)**：今ある葉のうち「分割したら一番lossが下がる葉」を毎回1個だけ伸ばす。木はいびつ＝旨い所だけ深掘り。
- **効く理由**：同じ葉数でより多くlossを削れる＝精度が出やすい。
- **危ない理由**：一点集中で深掘り→過学習しやすい。歯止めは `max_depth` ではなく **`num_leaves`（葉の総数の上限）**。

下で「木の本数 vs 精度」の過学習カーブを引く（Day5のGBDT版と同型）。

In [ ]:
# 足場：木の本数 vs train/test 精度（過学習カーブ）
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
N = 800
m = LGBMClassifier(n_estimators=N, learning_rate=0.1, random_state=0, verbose=-1).fit(Xtr, ytr)

ks = list(range(1, N + 1, 5))
tr_acc = [accuracy_score(ytr, m.predict(Xtr, num_iteration=k)) for k in ks]
te_acc = [accuracy_score(yte, m.predict(Xte, num_iteration=k)) for k in ks]
bi = int(np.argmax(te_acc))
print(f'test ピーク {te_acc[bi]:.3f} @ {ks[bi]}本 / train@800={tr_acc[-1]:.3f} / test@800={te_acc[-1]:.3f}')

plt.figure(figsize=(8, 5))
plt.plot(ks, tr_acc, label='train'); plt.plot(ks, te_acc, label='test')
plt.axvline(ks[bi], color='gray', ls='--', alpha=.6, label=f'test peak @ {ks[bi]}')
plt.xlabel('n_estimators (number of trees)'); plt.ylabel('accuracy')
plt.title('LightGBM: trees vs accuracy (lr=0.1, num_leaves=31)')
plt.legend(); plt.grid(alpha=.3)
plt.savefig('lgbm_overfit.png', dpi=100, bbox_inches='tight'); plt.close()
# => 伸びは150本前後で頭打ち。train は201本で1.000(丸暗記)、test は0.78で頭打ち＝過学習(Day3)。

![LightGBM 過学習カーブ](lgbm_overfit.png)

青(train)はどんどん上がって1.0に張り付く（丸暗記）のに、オレンジ(test=汎化)は150本で頭打ち。**青とオレンジの開き＝過学習のすき間**。201本以降に積んだ木は test を伸ばさず、train の暗記に注がれただけ。

## 武器③：early stopping ＝ 「木は何本？」を自動で決める

- **発想の転換**：`n_estimators` は当てにいく値ではなく **天井**（例 2000）。止め時は機械に任せる。
- **何を見張る？** train はすぐ 1.0 で天井＝止め時を教えない。だから **validation（模試）** の点を見張る。
- **なぜ test を見ちゃダメ？** test を見ながら止め時を選ぶと、その特定データに“たまたまハマる”点を選んでしまい、test 点が“盛られた数字”になる＝**リーク**。だから train / validation / test の **3分割**：train で学習、validation で止め時判断、test は最後の正直な採点用に触らない。

In [ ]:
# 本筋③：early stopping（3分割 → validation を見張って自動停止）
from lightgbm import early_stopping

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0, stratify=y_tmp)

model = LGBMClassifier(n_estimators=2000, learning_rate=0.1, random_state=0, verbose=-1)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],                                       # 見張る模試データ
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)],   # 50回伸びなきゃ止める
)
print(f'天井=2000 に対し選ばれた本数: {model.best_iteration_} 本')
print(f'触ってない test での精度: {accuracy_score(y_te, model.predict(X_te)):.3f}')
# => 106本で自動停止 / test 0.760。カーブの“膝(150本前後)”を機械が自力で当てた。

In [ ]:
# 足場：我慢の回数(stopping_rounds)を変えると？
print('stopping_rounds | 止まった本数 | test精度')
for p in [5, 20, 50, 200]:
    mm = LGBMClassifier(n_estimators=2000, learning_rate=0.1, random_state=0, verbose=-1)
    mm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
           callbacks=[early_stopping(stopping_rounds=p, verbose=False)])
    acc = accuracy_score(y_te, mm.predict(X_te))
    print(f'   {p:4d}         |   {mm.best_iteration_:4d} 本    |  {acc:.3f}')
# => 5:22本/0.708(早すぎ＝学習不足)  20/50/200:106本/0.760(十分なら安定)
#    教訓：stopping_rounds はケチらず多め(50が定番)。唯一の危険はケチること。

## Day7 まとめ

- **3つの武器**：① histogram（255bin＝速さ、実測85倍）／② leaf-wise（best-first＝旨い葉だけ深掘り、`num_leaves`で制御）／③ early stopping（validation見張りで本数を自動決定）。
- **生のLightGBMは精度の魔法ではない**（同条件でGBDTと同じ土俵）。強さの正体は **速さ → 試行回数 → そこで精度を稼ぐ**。
- **リーク**：止め時の判断に test を覗くと物差しが嘘をつく → train/valid/test の3分割。
- **実務でいつ使う？** 表(テーブル)データで、ある列を他の列から予測したいとき（離脱予測・課金予測・不正検知・需要予測…）。画像/音声/テキストはディープ、**表ならまずLightGBM**。

### 次回の候補
- `num_leaves` 等のチューニングで GBDT 0.806 を実際に超えにいく
- Kaggle（Titanic）を LightGBM + 特徴量 + early stopping で最初から最後まで一周（実務の形）
- 評価指標の使い分け（accuracyだけでない：precision/recall/AUC）